# 02 - Preprocess Sentiment Data

Creates the cleaned sentiment dataset and fixed train, validation, and test splits used by notebook 03. Run this notebook before training.

In [ ]:
from pathlib import Path
import html
import json
import re
import unicodedata

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
EXPECTED_LABELS = {'negative', 'neutral', 'positive'}
AI_DIR = Path.cwd().resolve().parent
RAW_DATA_PATH = AI_DIR / 'datasets' / 'student_feedback_sentiment_dataset.csv'
PROCESSED_DIR = AI_DIR / 'datasets' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def clean_feedback_text(value: object) -> str:
    """Apply minimal, meaning-preserving cleanup for sentiment classification."""
    text = html.unescape(str(value))
    text = unicodedata.normalize('NFKC', text).lower()
    text = text.replace('’', "'")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw_df = pd.read_csv(RAW_DATA_PATH)
required_columns = {'feedback_text', 'sentiment'}
if not required_columns.issubset(raw_df.columns):
    raise ValueError(f'Expected columns {sorted(required_columns)}, got {raw_df.columns.tolist()}')

df = raw_df.loc[:, ['feedback_text', 'sentiment']].copy()
df['sentiment'] = df['sentiment'].astype('string').str.strip().str.lower()
df = df.dropna(subset=['feedback_text', 'sentiment'])
df['feedback_text'] = df['feedback_text'].map(clean_feedback_text)
df = df.loc[df['feedback_text'].ne('')].copy()
df = df.loc[df['sentiment'].isin(EXPECTED_LABELS)].copy()
df = df.drop_duplicates(subset=['feedback_text', 'sentiment']).reset_index(drop=True)

if set(df['sentiment'].unique()) != EXPECTED_LABELS:
    raise ValueError(f'Unexpected labels after cleaning: {sorted(df["sentiment"].unique())}')

df['text_length'] = df['feedback_text'].str.len()
df['word_count'] = df['feedback_text'].str.split().str.len()
df.head()

In [ ]:
print(f'Raw rows: {len(raw_df):,}')
print(f'Clean rows: {len(df):,}')
print(f'Removed rows: {len(raw_df) - len(df):,}')
display(df['sentiment'].value_counts().rename_axis('sentiment').to_frame('count'))
display(df[['text_length', 'word_count']].describe())

In [ ]:
# 80% train, 10% validation, 10% test. Stratification preserves label balance.
train_df, holdout_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=df['sentiment']
)
validation_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=RANDOM_STATE, stratify=holdout_df['sentiment']
)

splits = {'train': train_df, 'validation': validation_df, 'test': test_df}
for split_name, split_df in splits.items():
    split_path = PROCESSED_DIR / f'sentiment_{split_name}.csv'
    split_df[['feedback_text', 'sentiment']].sort_index().to_csv(split_path, index=False)

clean_path = PROCESSED_DIR / 'sentiment_clean.csv'
df[['feedback_text', 'sentiment']].to_csv(clean_path, index=False)

summary = pd.DataFrame({
    name: part['sentiment'].value_counts().reindex(sorted(EXPECTED_LABELS), fill_value=0)
    for name, part in splits.items()
}).T
summary['total'] = summary.sum(axis=1)
display(summary)

In [ ]:
manifest = {
    'random_state': RANDOM_STATE,
    'source_file': str(RAW_DATA_PATH.relative_to(AI_DIR)),
    'cleaning': 'HTML unescape, Unicode normalization, lowercase, whitespace normalization, and exact text-label deduplication.',
    'labels': sorted(EXPECTED_LABELS),
    'split_rows': {name: len(part) for name, part in splits.items()},
}
manifest_path = PROCESSED_DIR / 'sentiment_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print(f'Wrote cleaned data and splits to: {PROCESSED_DIR}')